# NLA Steering — GPU runner

Единственный ноутбук проекта. Запусти ячейки сверху вниз **один раз** за сессию
и оставь последнюю работать: она забирает задачи из очереди на Drive,
выполняет их и складывает результаты обратно.

Runtime → Change runtime type → **T4 GPU**.

Что нужно один раз настроить: в панели слева 🔑 Secrets добавить `HF_TOKEN`
и включить для него доступ этому ноутбуку.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
import os, subprocess, sys
from pathlib import Path

REPO_URL = "https://github.com/xatxor/NLA_Steering_DLS.git"
REPO = Path('/content/repo')

if REPO.exists():
    subprocess.run(['git', '-C', str(REPO), 'fetch', '--all'], check=True)
    subprocess.run(['git', '-C', str(REPO), 'reset', '--hard', 'origin/main'], check=True)
else:
    subprocess.run(['git', 'clone', REPO_URL, str(REPO)], check=True)

# Токен из Colab Secrets -> в окружение, чтобы huggingface_hub его подхватил.
from google.colab import userdata
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')

# Кэш HF на локальный диск инстанса, а не на Drive: Drive медленный на
# множестве мелких файлов и его квота нам дороже, чем эфемерные 100 ГБ Colab.
os.environ['HF_HOME'] = '/content/hf_cache'

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'transformers>=4.46', 'accelerate>=1.0', 'datasets>=3.0',
                'sentence-transformers>=3.0', 'python-dotenv', 'pyyaml', 'safetensors'],
               check=True)

sys.path.insert(0, str(REPO / 'src'))
from nla_steering.paths import ensure_layout, workspace
ensure_layout()
print('репо   :', subprocess.run(['git', '-C', str(REPO), 'log', '--oneline', '-1'],
                                 capture_output=True, text=True).stdout.strip())
print('рабочая:', workspace())


In [ ]:
import torch

print('torch      :', torch.__version__)
print('cuda       :', torch.cuda.is_available())
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print('gpu        :', props.name)
    print('память     : %.1f ГБ' % (props.total_memory / 1024**3))
    print('capability :', f'{props.major}.{props.minor}')
    # bf16 требует Ampere (8.0+). На T4 (7.5) считаем в fp16 и без flash-attn 2.
    print('bf16       :', torch.cuda.is_bf16_supported())


In [ ]:
import subprocess, time, traceback
from datetime import datetime, timezone

from nla_steering import jobs

POLL = 15  # секунд


def run_job(job, job_path):
    log = jobs.log_path(job.id)
    with log.open('w', encoding='utf-8', buffering=1) as fh:
        def emit(line):
            print(line, end='')
            fh.write(line)

        emit(f'=== {job.id}\n{job.note}\n{job.command()}\n\n')
        subprocess.run(['git', '-C', str(REPO), 'pull', '--ff-only'],
                       capture_output=True, text=True)
        proc = subprocess.Popen(
            [sys.executable, job.script, *job.args],
            cwd=REPO, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
            text=True, bufsize=1,
            env={**os.environ, 'PYTHONPATH': str(REPO / 'src'), 'PYTHONUNBUFFERED': '1'},
        )
        for line in proc.stdout:
            emit(line)
        code = proc.wait()
        emit(f'\n=== exit {code}\n')
    jobs.finish(job_path, ok=(code == 0))
    return code


print('воркер запущен, жду задачи. Останов — кнопка stop.')
idle_since = time.monotonic()
while True:
    try:
        claimed = jobs.claim()
        if claimed is None:
            if time.monotonic() - idle_since > 300:
                print(f'{datetime.now(timezone.utc):%H:%M:%S} простой, очередь пуста')
                idle_since = time.monotonic()
            time.sleep(POLL)
            continue
        idle_since = time.monotonic()
        job, job_path = claimed
        print(f'\n>>> {job.id}')
        code = run_job(job, job_path)
        print(f'<<< {job.id} exit={code}')
    except KeyboardInterrupt:
        print('остановлен')
        break
    except Exception:
        traceback.print_exc()
        time.sleep(POLL)
